# Lufthansa API — Exploration
Testing the LH API client in mock mode (no token needed).

In [ ]:
import sys
sys.path.append('.')

from lufthansa_api.client import LufthansaAPIClient

client = LufthansaAPIClient(use_mock=True)
print('Client erstellt!')

## 1. Flughäfen abrufen

In [ ]:
airports = client.get_airports()

for airport in airports['ResourceResponse']['Airport']:
    code = airport['AirportCode']
    name = airport['Names']['Name'][0]['$']
    country = airport['CountryCode']
    print(f'{code} | {name} | {country}')

## 2. Airlines abrufen

In [ ]:
airlines = client.get_airlines()

for airline in airlines['ResourceResponse']['Airline']:
    code = airline['AirlineCode']
    name = airline['Names']['Name'][0]['$']
    print(f'{code} | {name}')

## 3. Einzelnen Flughafen abfragen

In [ ]:
response = client.get_airports(airport_code='BER')
airport = response['ResourceResponse']['Airport'][0]

print('Code:   ', airport['AirportCode'])
print('City:   ', airport['CityCode'])
print('Country:', airport['CountryCode'])
coord = airport['Position']['Coordinate']
print('Coords: ', coord['Latitude'], coord['Longitude'])

## 4. Daten in PostgreSQL schreiben

In [ ]:
from db.postgres.connector import from_env

airports_data = client.get_airports()['ResourceResponse']['Airport']
airlines_data = client.get_airlines()['ResourceResponse']['Airline']

with from_env() as db:
    db.create_tables()
    db.insert_airports(airports_data)
    db.insert_airlines(airlines_data)
    print(f'{len(airports_data)} airports und {len(airlines_data)} airlines gespeichert')

## 5. Kontrolle — aus DB lesen

In [ ]:
with from_env() as db:
    db.cursor.execute('SELECT code, name, country_code FROM airports')
    print('Airports:')
    for row in db.cursor.fetchall():
        print(f'  {row[0]} | {row[1]} | {row[2]}')

    db.cursor.execute('SELECT code, name FROM airlines')
    print('Airlines:')
    for row in db.cursor.fetchall():
        print(f'  {row[0]} | {row[1]}')